# Système d'assistance visuelle en temps réel — YOLOv8

**Objectif :** analyser le flux de la webcam en direct avec YOLOv8 pour :
- compter le nombre de personnes visibles à l'écran
- déclencher une alerte visuelle si un objet cible est détecté

**Stack :** Python, Ultralytics (YOLOv8), OpenCV

## 1. Imports et chargement du modèle

On charge une seule fois le modèle pré-entraîné `yolov8n.pt` (nano, léger et rapide, entraîné sur les 80 classes du dataset COCO).

In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO("yolov8n.pt")
print("Modèle chargé —", len(model.names), "classes disponibles")

## 2. Test rapide sur une image fixe

Avant de lancer la webcam, on vérifie que le modèle fonctionne sur une image simple (capture unique de la webcam).

In [ ]:
cap = cv2.VideoCapture(0)
ret, frame = cap.read()
cap.release()

if ret:
    results = model(frame)
    annotated = results[0].plot()
    cv2.imwrite("test_frame.jpg", annotated)
    print("Image annotée sauvegardée : test_frame.jpg")
else:
    print("Webcam non accessible")

## 3. Logique métier : compteur + alerte

Pour chaque détection, on récupère sa classe via `model.names[class_id]`. On incrémente un compteur si c'est une `"person"`, et on déclenche une alerte si l'objet cible (`TARGET_CLASS`) apparaît.

In [ ]:
TARGET_CLASS = "knife"  # objet à surveiller — change selon ton besoin

def analyser_frame(frame):
    results = model(frame, verbose=False)
    boxes = results[0].boxes

    person_count = 0
    alert_triggered = False

    for box in boxes:
        class_id = int(box.cls[0])
        class_name = model.names[class_id]

        if class_name == "person":
            person_count += 1
        if class_name == TARGET_CLASS:
            alert_triggered = True

    annotated = results[0].plot()
    cv2.putText(annotated, f"Personnes: {person_count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    if alert_triggered:
        cv2.putText(annotated, "ALERTE", (50, 100),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 3)

    return annotated, person_count, alert_triggered

## 4. Boucle temps réel (webcam)

Lance le flux en direct. Appuie sur **`q`** dans la fenêtre vidéo pour arrêter proprement.

In [ ]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    annotated, person_count, alert_triggered = analyser_frame(frame)
    cv2.imshow("YOLOv8 - Assistance visuelle", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## 5. Conclusion

- Modèle utilisé : YOLOv8n (pré-entraîné sur COCO, 80 classes)
- Fonctionnalités : détection en temps réel, comptage de personnes, alerte visuelle sur objet cible
- Pistes d'amélioration : enregistrement vidéo des alertes, notification sonore, filtrage multi-classes, export des logs de détection